# Research: ML-Temporal-CNN

Temporal CNN Direction Prediction Strategy. This algorithm uses a Temporal Convolutional Neural Network to forecast the direction (up/down/stationary) of future stock prices:

**Calibration target**: `TemporalCNNPredictionAlgorithm` in `main.py`

In [1]:
# Initialize QuantBook
from datetime import datetime
from QuantConnect.Research import QuantBook

qb = QuantBook()
symbols = {}

symbols['QQQ'] = qb.add_equity('QQQ', Resolution.DAILY).symbol

start = datetime(2020, 1, 1)
end = datetime(2026, 1, 1)
history = qb.history(symbols['QQQ'], start, end, Resolution.DAILY)
print(f"History: {len(history)} rows")
if len(history) > 0:
    display(history.head())

History: 1508 rows


close        high         low        open  \
symbol time                                                                  
QQQ    2020-01-02 16:00:00  214.324953  214.324953  212.163460  212.579894   
       2020-01-03 16:00:00  212.361762  213.640811  211.469402  211.489232   
       2020-01-06 16:00:00  213.730046  213.759792  210.448146  210.696024   
       2020-01-07 16:00:00  213.700301  214.305123  213.026074  213.809367   
       2020-01-08 16:00:00  215.306549  216.288144  213.333442  213.670556   

                                volume  
symbol time                             
QQQ    2020-01-02 16:00:00  28920803.0  
       2020-01-03 16:00:00  24781183.0  
       2020-01-06 16:00:00  19869056.0  
       2020-01-07 16:00:00  20686605.0  
       2020-01-08 16:00:00  23897356.0

## 1. Data Exploration

Statistical overview: returns distribution, correlations, volatility.

In [2]:
import pandas as pd
import numpy as np

if len(history) == 0:
    print('WARNING: No history data available')
    closes = pd.DataFrame()
    returns = pd.DataFrame()
else:
    if isinstance(history.index, pd.MultiIndex):
        closes = history['close'].unstack(level=0)
    else:
        closes = history['close']
    returns = closes.pct_change().dropna()
    print("=== Returns Summary ===")
    print(returns.describe().round(4))
    print()
    print("=== Correlation Matrix ===")
    print(returns.corr().round(3))

=== Returns Summary ===
symbol        QQQ
count   1507.0000
mean       0.0003
std        0.0097
min       -0.1198
25%        0.0000
50%        0.0000
75%        0.0000
max        0.0847

=== Correlation Matrix ===
symbol  QQQ
symbol     
QQQ     1.0


## 2. Signal Analysis

Analyze temporal features for CNN/LSTM model inputs.

In [3]:
# Signal computation
if len(returns) == 0:
    print('No data for signal analysis')
    signal = pd.Series(dtype=float)
else:
    # Compute generic signals
    primary = closes.iloc[:, 0] if closes.shape[1] > 0 else closes
    returns_5d = primary.pct_change(5)
    vol_10d = primary.pct_change().rolling(10).std()
    signal = returns_5d.dropna()
    signal.name = 'return_5d'

print("=== Signal Statistics ===")
if len(signal) > 0:
    print(f"Signal mean: {signal.mean():.6f}")
    print(f"Signal std: {signal.std():.6f}")
    print(f"Signal range: [{signal.min():.6f}, {signal.max():.6f}]")

=== Signal Statistics ===
Signal mean: 0.001494
Signal std: 0.017882
Signal range: [-0.161964, 0.126599]


## 3. Parameter Sensitivity

Test different parameter values for universe_size.

In [4]:
# Parameter sensitivity scan
if len(returns) == 0:
    print('No data for parameter scan')
    results = []
    best_params = 'N/A'
else:
    # Parameter scan for universe_size
    scan_values = [5, 10, 20, 30, 60]
    results = []
    for val in scan_values:
        # Compute metric for each parameter value
        vol = closes.iloc[:, 0].pct_change().rolling(val).std().mean()
        results.append((f'universe_size={val}', vol))
    best_params = min(results, key=lambda x: x[1])[0]

print("=== Parameter Scan Results ===")
for params, metric in results:
    print(f"  {params}: metric={metric:.4f}")
print(f"\nBest: {best_params}")

=== Parameter Scan Results ===
  universe_size=5: metric=0.0036
  universe_size=10: metric=0.0037
  universe_size=20: metric=0.0038
  universe_size=30: metric=0.0039
  universe_size=60: metric=0.0039

Best: universe_size=5


## 4. Calibration to main.py

Mapping research findings to algorithm parameters:

| Parameter | Research Value | main.py Default |
|-----------|---------------|----------------|
| `universe_size` | (research value) | default |

In [5]:
# Calibration summary
calibration = {
    "universe_size": "<research_value>",
}

print("=== Calibration Parameters ===")
for k, v in calibration.items():
    print(f"  {k}: {v}")
print("\nTo apply: update get_parameter() defaults in main.py")

=== Calibration Parameters ===
  universe_size: <research_value>

To apply: update get_parameter() defaults in main.py


## 5. Conclusion & Next Iteration

**Findings:**
- Universe (QQQ) analyzed with deep_learning signals
- Parameter sensitivity for universe_size scanned

**Next iteration:**
- Test alternative universe composition
- Add regime-conditional analysis
- Validate against backtest results in main.py